# ARC-AGI-3 LLM Agent (Qwen2.5-7B NF4 for T4)

LLM agent using Qwen2.5-7B-Instruct with NF4 quantization via bitsandbytes.
Designed for Tesla T4 GPU (15GB VRAM).


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

print(f"TRUE_SUBMISSION={TRUE_SUBMISSION}")
print("GPU check:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"])

In [ ]:
# Install ARC runtime
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
    "--find-links", "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    "arc-agi", "python-dotenv",
], stdout=subprocess.DEVNULL)
print("ARC runtime installed")

# Check bitsandbytes
try:
    import bitsandbytes
    print(f"bitsandbytes already installed: {bitsandbytes.__version__}")
except ImportError:
    print("bitsandbytes not found, installing from dataset...")
    # Find bitsandbytes dataset
    bnb_path = None
    for p in Path("/kaggle/input").iterdir():
        if "bitsandbytes" in p.name.lower():
            bnb_path = p
            print(f"Found bitsandbytes at: {bnb_path}")
            break
    
    if bnb_path and bnb_path.exists():
        # Install from local wheel files
        wheels = list(bnb_path.glob("*.whl"))
        if wheels:
            print(f"Installing from {len(wheels)} wheel(s)...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install",
                "--no-index", "--no-deps",
                *[str(w) for w in wheels],
                "--quiet"
            ])
            import bitsandbytes
            print(f"bitsandbytes installed: {bitsandbytes.__version__}")
        else:
            # Try pip install --find-links
            subprocess.check_call([
                sys.executable, "-m", "pip", "install",
                "--no-index", "--find-links", str(bnb_path),
                "bitsandbytes", "--quiet"
            ])
            import bitsandbytes
            print(f"bitsandbytes installed: {bitsandbytes.__version__}")
    else:
        print("ERROR: bitsandbytes dataset not found!")
        raise RuntimeError("bitsandbytes not available")

In [ ]:
# Find model path
print("Searching for Qwen2.5-7B-Instruct...")
MODEL_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files:
        config_path = os.path.join(root, "config.json")
        try:
            cfg = json.load(open(config_path))
            if cfg.get("model_type") in ("qwen2", "qwen"):
                MODEL_PATH = Path(root)
                print(f"Found Qwen2 model at: {MODEL_PATH}")
                break
        except Exception:
            pass

if MODEL_PATH is None:
    for p in sorted(Path("/kaggle/input").iterdir()):
        if p.is_dir() and (p / "config.json").exists():
            MODEL_PATH = p
            print(f"Found model at: {MODEL_PATH}")
            break

if MODEL_PATH is None:
    MODEL_PATH = Path("/kaggle/input/qwen2-5-7b-instruct")
    print(f"Fallback: {MODEL_PATH}")

print(f"Model path: {MODEL_PATH}")
print(f"Exists: {MODEL_PATH.exists()}")
if MODEL_PATH.exists():
    files = sorted(f.name for f in MODEL_PATH.iterdir())
    print(f"Files ({len(files)}): {files[:15]}")
    total = sum(f.stat().st_size for f in MODEL_PATH.iterdir() if f.is_file())
    print(f"Total: {total / 1e9:.1f} GB")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Configuring NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading model with NF4...")
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

print("\nModel loaded! GPU memory:")
subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader"])
print(f"Config: {model.config.num_hidden_layers} layers, {model.config.hidden_size} hidden")

# Smoke test
print("\nSmoke test...")
test_input = tokenizer("Hello!", return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**test_input, max_new_tokens=15, do_sample=False)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Response: {response!r}")
print("\nMODEL READY!")

In [ ]:
# Deploy agent code (base64 encoded)
import base64
agent_b64 = "..."
agent_code = base64.b64decode(agent_b64).decode()
agent_dir = Path("/tmp/arc_llm_t4")
agent_dir.mkdir(parents=True, exist_ok=True)
(agent_dir / "__init__.py").write_text("")
(agent_dir / "llm_agent.py").write_text(agent_code)
print(f"Agent deployed to {agent_dir} ({len(agent_code)} chars)")
# Verify it compiles
compile(agent_code, "llm_agent.py", "exec")
print("Agent code compiles OK")

In [ ]:
import torch
import shutil
import urllib.request

if TRUE_SUBMISSION:
    framework_src = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    framework = WORKING / "ARC-AGI-3-Agents"
    if framework.exists():
        shutil.rmtree(framework)
    shutil.copytree(framework_src, framework)

    target = framework / "agents" / "templates"
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path("/tmp/arc_llm_t4/llm_agent.py"), target / "llm_agent.py")
    (target / "__init__.py").write_text("")

    # Register agent
    NEWLINE = chr(10)
    init_py = NEWLINE.join([
        '"""Agent registry."""',
        'from typing import Type',
        'from dotenv import load_dotenv',
        'from .agent import Agent, Playback',
        'from .recorder import Recorder',
        'from .swarm import Swarm',
        'from .templates.random_agent import Random',
        'from .templates.my_agent import MyAgent',
        'from .templates.llm_agent import LLMAgent',
        '',
        'load_dotenv()',
        'AVAILABLE_AGENTS = {"random": Random, "myagent": MyAgent, "llmagent": LLMAgent}',
        'for recording in Recorder.list():',
        '    AVAILABLE_AGENTS[recording] = Playback',
    ]) + NEWLINE
    (framework / "agents" / "__init__.py").write_text(init_py)

    env_content = "\n".join([
        "SCHEME=http", "HOST=gateway", "PORT=8001",
        "ARC_API_KEY=competition-gateway",
        "ARC_BASE_URL=http://gateway:8001/",
        "OPERATION_MODE=online", "ENVIRONMENTS_DIR=",
        f"RECORDINGS_DIR={WORKING}/recordings",
    ]) + "\n"
    (framework / ".env").write_text(env_content)

    # Set model on agent class
    sys.path.insert(0, str(framework))
    from agents.templates.llm_agent import LLMAgent as AgentClass
    AgentClass.set_model(model, tokenizer)

    # Wait for gateway
    print("Waiting for gateway...")
    deadline = time.time() + 600
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://gateway:8001/api/games", timeout=10) as r:
                if r.status < 500:
                    break
        except Exception:
            pass
        time.sleep(5)

    run_env = os.environ.copy()
    run_env["MPLBACKEND"] = "Agg"
    run_env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

    print("Starting LLM agent run...")
    result = subprocess.run(
        [sys.executable, "main.py", "--agent", "llmagent"],
        cwd=framework, env=run_env,
        capture_output=True, text=True, timeout=32000,
    )
    print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
    print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")
    print(f"Return code: {result.returncode}")
else:
    import pandas as pd
    placeholder = pd.DataFrame(
        [["placeholder_0", "placeholder", True, 0.0]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    placeholder.to_parquet(WORKING / "submission.parquet", index=False)
    print("Wrote save-phase placeholder submission.parquet")

In [ ]:
elapsed = time.time() - NOTEBOOK_START
print(f"Notebook elapsed: {elapsed:.0f}s ({elapsed/60:.1f} min)")